# Hypothetical accounting scenarios

Stress-test assumptions are not predicted outcomes, probabilities, savings or financial profitability.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
root=Path.cwd()
if root.name=='notebooks': root=root.parent
sys.path.insert(0,str(root))
from src.decision_features import configuration
from src.prioritization import priority_scores
from src.scenario_engine import accounting, break_even, mix_simulation
cfg=configuration(root)
def table(name): return pd.read_parquet(root/f'outputs/phase5/tables/{name}.parquet')
def chart(name): display(Image(filename=str(root/f'outputs/phase5/charts/{name}.png')))


## Observed and forecast baselines

Observed baseline: 2025 weighted NIC/item and item total. Planning baseline: frozen January–March 2026 point forecasts using the December 2025 observation cutoff. Historical 2025 cost ratios are held fixed as assumptions, not extrapolated trends.

In [ ]:
display(table('scenario_assumptions').head(12))
display(table('scenario_results').query("baseline_type == 'Model forecast baseline'").head())
n=table('national_decision_summary').iloc[0]

## One-way and combined sensitivities

NIC = items × weighted NIC/item. Volume and cost adjustments multiply, including their interaction.

In [ ]:
display({'demonstration_plus_5pct_volume':accounting(n.recent_items,n.recent_nic_per_item_gbp,5,0)})
display(table('scenario_results').query("scenario_name.str.startswith('C:')",engine='python').head())
chart('13_volume_sensitivity')
chart('14_cost_sensitivity')
chart('15_two_way_sensitivity')

## Mix-only conservation

Transfer 1/2/5 percentage points from the largest 2025 presentation to the second-largest. All 17 shares remain nonnegative and sum to one. Total prescription items stay fixed. This accounting simulation is not clinically appropriate substitution advice.

In [ ]:
p=table('presentation_decision_features').sort_values(['recent_items','presentation_key'],ascending=[False,True])
donor,receiver=p.presentation_key.iloc[:2]
demo=mix_simulation(p,n.recent_items,donor,receiver,1)
assert abs(demo.hypothetical_items.sum()-n.recent_items)<1e-6
display(table('presentation_mix_scenarios').head(17))
chart('16_mix_sensitivity')

## Regional and forecast planning sensitivity

Regional scenarios change one region at a time and leave the other six fixed. Forecast planning scenarios change point expectations hypothetically; these are not confidence bands.

In [ ]:
display(table('scenario_results').query("scenario_name.str.startswith('E:')",engine='python').head(10))
display(table('scenario_results').query("scenario_name.str.startswith('F:')",engine='python'))

## Accounting break-even and limits

Offsetting NIC/item change = 1/(1 + volume change) − 1. Equality of accounting NIC is unrelated to financial profitability. No intervention outcome is estimated.

In [ ]:
display({'plus_5pct_volume_offset_pct':break_even(5)})
display(table('accounting_break_even_analysis'))
chart('17_accounting_break_even')
checks=pd.read_csv(root/'outputs/phase5/validation/decision_reconciliation.csv')
assert checks.status.eq('passed').all()
display(checks.status.value_counts())